## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Aiden Zhu

**Date:** 9/30/2026

In [2]:
# Your Phase 1 solution to the problem goes here.
print("1) Regression is used to predict a numerical target while classification is used to predict a categorical or qualitative. In the example used in the lecture, regression could predict prices while classification could predict survival.")
print("2) A confusion table measures actual classes and predicted classes against each other. This allows us to understand the correctness of a model and determine the amount of correct predictions, or how well the model performs.")
print("3) SSE stands for Sum of Squared Error, and it calculates the prediction error of a validation set. A lower validation SSE means that the validation set has less errors.")
print("4) Overfitting and underfitting is determined by the value of k, which determines the number of close 'neighbors' that the model will look at to make a prediction. A high k value will create more overfit noise in the training data while a low k value could underfit noise and make predictions more similar.")
print("5) By splitting the data into training and testing sets, we can adjust models while comparing their predictions/outcomes on the testing set to evaluate model performance. We calculate the SSE or accuracy with different k values because it can show us the different training and validation errors for different k values and allows us to choose the lowest SSE value.")
print ("6) From the slides and vehicle example, I believe the advantage to using class labels is that we can see the actual vs predicted values from a clear numerical standpoint within each class, and it is easy to interpret individual predictions. The weakness is that it could be hard to interpret the overall model as the number of classes get larger. Also, it is a binary system that calculates only correct vs incorrect and doesn't include any confidence. The strength of a probability/prediction distribution is that it shows the confidence of the model rather than the binary right vs wrong. I think a weakness is that the distribution needs to be interpreted by people rather than straight up showing the right/wrong data, so conclusions can vary between people ")


1) Regression is used to predict a numerical target while classification is used to predict a categorical or qualitative. In the example used in the lecture, regression could predict prices while classification could predict survival.
2) A confusion table measures actual classes and predicted classes against each other. This allows us to understand the correctness of a model and determine the amount of correct predictions, or how well the model performs.
3) SSE stands for Sum of Squared Error, and it calculates the prediction error of a validation set. A lower validation SSE means that the validation set has less errors.
4) Overfitting and underfitting is determined by the value of k, which determines the number of close 'neighbors' that the model will look at to make a prediction. A high k value will create more overfit noise in the training data while a low k value could underfit noise and make predictions more similar.
5) By splitting the data into training and testing sets, we can 

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler       
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/land_mines.csv")
df.info()
print(df.isna().sum())
df.describe()
print(df["mine_type"].value_counts())
print("mine_type is a categorical variable with categories 1, 2, 3, 4, and 5. I was confused at first because the types are integers, but they are just labels for the mines. If they were called something else it would've been easier to differentiate. voltage, height, and soil are numbers between 0-1.")

eval_X_raw = df[["voltage", "height", "soil"]]
eval_y = df["mine_type"]
(
    eval_X_train_raw, 
    eval_X_valid_raw, 
    eval_y_train, 
    eval_y_valid 
) = train_test_split(
    eval_X_raw, eval_y, test_size=0.50, random_state=65
)

scaler = MinMaxScaler()
eval_X_train_scaled = scaler.fit_transform(eval_X_train_raw)
eval_X_valid_scaled = scaler.transform(eval_X_valid_raw)

eval_ks = np.arange(1, 51)
eval_valid_error = []
eval_train_error = []
for eval_k in eval_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(eval_X_train_scaled, eval_y_train)
    eval_valid_hat = eval_candidate.predict(eval_X_valid_scaled)
    eval_train_hat = eval_candidate.predict(eval_X_train_scaled)
    eval_valid_error.append(np.mean(
        eval_y_valid.to_numpy() != eval_valid_hat
    ))
    eval_train_error.append(np.mean(
        eval_y_train.to_numpy() != eval_train_hat
    ))
eval_k_star = int(eval_ks[np.argmin(eval_valid_error)])
print(eval_k_star)
print()

print("To select k, I wanted to find the lowest error. I used error rate instead of SSE because mine_type is a categorical variable and not numerical. Mimicking the slides, I used k values from 1-50 on the training set then calculated the error rate on the validation set. To find the k with the lowest error, I used np.argmin like the slides did to find k=2 with a 45% accuracy rate. ")



model = KNeighborsClassifier(n_neighbors=eval_k_star).fit(eval_X_train_scaled, eval_y_train)
y_prediction = model.predict(eval_X_valid_scaled)

validation_set = eval_X_valid_raw.copy()
validation_set["actual"] = eval_y_valid
validation_set["predicted"] = y_prediction

for x in np.arange(1, 6):
    print(f"Actual type {x}:")
    print(validation_set.loc[validation_set["actual"] == x, "predicted"].value_counts())

print(1 - eval_valid_error[eval_k_star - 1])

print("The model has an accuracy of 45% and classifies 76 of 169 correctly. The performance is more accurate for mine_type 2 and less accurate for mine_type 4 and 5.")
print("The model is definitely better at predicted specific mine types than others, such as mine_type 2. Also, the model predicts mine_type 1 right many times, but then it mislabels as another mine_type so it gets it wrong. As a result, I wouldn't recommend using this model to make important final decisions because of its low accuracy on other types besides 2, but it performs pretty well for mine type 2. I would advise someone to not use the model until it is trained better, but it could be used to amke broad predictions because of its 45% accuracy in the meantime.")






<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
voltage      0
height       0
soil         0
mine_type    0
dtype: int64
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
2

To select k, I wanted to find the lowest error. Mimicking the slides, I used k values from 1-50 on the training set then calculated the error rate on the validation set. To find the k with the lowest error, I used np.argmin like the slides did to find k=2 with a 45% accuracy rate. 
Actual type 1:
predicted
1    21
3     6
4     4
5     2
2     1
Name: count, dtype: int64
Actual type 2:
predicted
2    33
4     3
3     2
Name: count, dtype: int64
Actual type 3:


## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** Claude Opus 5.5

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]